In [1]:
import numpy as np 
import pandas as pd  
from pathlib import Path 

In [3]:
CLEANED_DATAFRAME_LOCATION = Path(r'..\datasets\cleaned_dataset.csv')
cleaned_df = pd.read_csv(CLEANED_DATAFRAME_LOCATION)

# add new columns to division dataframe
division_df = cleaned_df.groupby(['Division'])[['Sales', 'Units', 'Gross Profit', 'Cost']].sum()

division_df['Average Margin'] = division_df['Gross Profit'] / division_df['Sales']
division_df['Revenue'] = division_df['Sales'] / division_df['Sales'].sum()
division_df['Profit Contribution'] = division_df['Gross Profit'] / division_df['Gross Profit'].sum()
division_df['Rev-to-Profit Ratio'] = (division_df['Profit Contribution'] / division_df['Revenue'])
division_df['Cost Efficiency'] = division_df['Gross Profit'] / division_df['Cost']

overall_avg_margin = division_df['Gross Profit'].sum() / division_df['Sales'].sum()
division_df['Margin Efficiency Score'] = (division_df['Average Margin'] / overall_avg_margin)
division_df['Rev-Profit Gap'] = (division_df['Profit Contribution'] - division_df['Revenue'])

In [4]:
division_df

,Sales,Units,Gross Profit,Cost,Average Margin,Revenue,Profit Contribution,Rev-to-Profit Ratio,Cost Efficiency,Margin Efficiency Score,Rev-Profit Gap
Division,,,,,,,,,,,
Chocolate,131692.90,37275,88824.62,42868.28,0.674483,0.928830,0.950577,1.023414,2.072036,1.023414,0.021747
Other,9663.25,1242,4333.45,5329.80,0.448446,0.068155,0.046375,0.680442,0.813061,0.680442,-0.021779
Sugar,427.48,137,284.73,142.75,0.666066,0.003015,0.003047,1.010643,1.994606,1.010643,0.000032


---
### Division Profit Calculation
---

- Aggregate metrics by Division
- Compare:
    - Average margin by division
    - Revenue vs profit imbalance
- Identify divisions with:
    - Strong financial efficiency
    - Structural margin issues

In [11]:
## -------------AVERAGE PROFIT MARGIN BY DIVISION----------------
print("="*60, "Compare Average Profit Margin by Division", "="*60, sep='\n')
print((division_df[['Average Margin']]*100).round(2))

## ----------------REVENUE VS PROFIT IMBALANCE-------------------
print("="*60, "Revenue vs Profit Imbalance", "="*60, sep='\n')
print((division_df[['Rev-to-Profit Ratio', 'Rev-Profit Gap']]))

print("\n")
print("=" * 80)
print("DIVISIONS WITH STRONG FINANCIAL EFFICIENCY")
print("=" * 80)
print("Criteria (at least 2 of 3 must be met):")
print("  1. Profit share > Revenue share  (Rev-to-Profit Ratio > 1.0)")
print("  2. Above-average margin          (Margin Efficiency Score > 1.0)")
print("  3. Cost efficiency above median  (Cost Efficiency > median)")
print()

median_cost_eff = division_df['Cost Efficiency'].median()
print(f"Median Cost Efficiency: {median_cost_eff:.4f}\n")

# Boolean flags for each criterion
division_df['Profit > Revenue Share'] = division_df['Rev-to-Profit Ratio'] > 1.0
division_df['Above Avg Margin'] = division_df['Margin Efficiency Score'] > 1.0
division_df['High Cost Efficiency'] = division_df['Cost Efficiency'] > median_cost_eff

# Count how many criteria each division meets
efficiency_cols = ['Profit > Revenue Share', 'Above Avg Margin', 'High Cost Efficiency']
division_df['Efficiency Score (out of 3)'] = division_df[efficiency_cols].sum(axis=1)

# Strong financial efficiency = meets at least 2 of 3 criteria
strong_efficiency = division_df[division_df['Efficiency Score (out of 3)'] >= 2]

if strong_efficiency.empty:
    print("No divisions meet the strong efficiency threshold.")
else:
    for div in strong_efficiency.index:
        row = strong_efficiency.loc[div]
        print(f"  >> {div}")
        print(f"     * Average Margin:       {row['Average Margin']:.2%}")
        print(f"     * Revenue Share:        {row['Revenue']:.2%}")
        print(f"     * Profit Contribution:  {row['Profit Contribution']:.2%}")
        print(f"     * Rev-to-Profit Ratio:  {row['Rev-to-Profit Ratio']:.4f}")
        print(f"     * Cost Efficiency:      {row['Cost Efficiency']:.4f}")
        met = [col for col in efficiency_cols if row[col]]
        print(f"     * Criteria Met ({int(row['Efficiency Score (out of 3)'])}/"
              f"3): {', '.join(met)}")
        print()

print("=" * 80)
print("DIVISIONS WITH STRUCTURAL MARGIN ISSUES")
print("=" * 80)
print("Criteria (at least 1 of 3 indicates a structural issue):")
print("  1. Revenue share > Profit share  (negative Rev-Profit Gap)")
print("  2. Below-average margin          (Margin Efficiency Score < 1.0)")
print("  3. Low cost efficiency           (Cost Efficiency < median)")
print()

# Boolean flags for structural issues
division_df['Revenue > Profit Share'] = division_df['Rev-Profit Gap'] < 0
division_df['Below Avg Margin'] = division_df['Margin Efficiency Score'] < 1.0
division_df['Low Cost Efficiency'] = division_df['Cost Efficiency'] < median_cost_eff

issue_cols = ['Revenue > Profit Share', 'Below Avg Margin', 'Low Cost Efficiency']
division_df['Margin Issue Score (out of 3)'] = division_df[issue_cols].sum(axis=1)

# Structural margin issues = meets at least 1 of 3 issue criteria
margin_issues = division_df[division_df['Margin Issue Score (out of 3)'] >= 1]

if margin_issues.empty:
    print("No divisions exhibit structural margin issues.")
else:
    for div in margin_issues.index:
        row = margin_issues.loc[div]
        print(f"  >> {div}")
        print(f"     * Average Margin:          {row['Average Margin']:.2%}")
        print(f"     * Revenue Share:           {row['Revenue']:.2%}")
        print(f"     * Profit Contribution:     {row['Profit Contribution']:.2%}")
        print(f"     * Rev-Profit Gap:          {row['Rev-Profit Gap']:+.4f}")
        print(f"     * Cost Efficiency:         {row['Cost Efficiency']:.4f}")
        print(f"     * Margin Efficiency Score: {row['Margin Efficiency Score']:.4f}")
        issues = [col for col in issue_cols if row[col]]
        print(f"     * Issues Found ({len(issues)}/3): {', '.join(issues)}")
        print()

Compare Average Profit Margin by Division
           Average Margin
Division                 
Chocolate           67.45
Other               44.84
Sugar               66.61
Revenue vs Profit Imbalance
           Rev-to-Profit Ratio  Rev-Profit Gap
Division                                      
Chocolate             1.023414        0.021747
Other                 0.680442       -0.021779
Sugar                 1.010643        0.000032


DIVISIONS WITH STRONG FINANCIAL EFFICIENCY
Criteria (at least 2 of 3 must be met):
  1. Profit share > Revenue share  (Rev-to-Profit Ratio > 1.0)
  2. Above-average margin          (Margin Efficiency Score > 1.0)
  3. Cost efficiency above median  (Cost Efficiency > median)

Median Cost Efficiency: 1.9946

  >> Chocolate
     * Average Margin:       67.45%
     * Revenue Share:        92.88%
     * Profit Contribution:  95.06%
     * Rev-to-Profit Ratio:  1.0234
     * Cost Efficiency:      2.0720
     * Criteria Met (3/3): Profit > Revenue Share, Above Avg

In [12]:
print("=" * 80)
print("DIVISION FINANCIAL HEALTH SUMMARY")
print("=" * 80)


def classify_division(row):
    """Classify a division based on efficiency and margin issue scores."""
    eff = row['Efficiency Score (out of 3)']
    issues = row['Margin Issue Score (out of 3)']

    if eff >= 2 and issues == 0:
        return 'Strong Financial Efficiency'
    elif issues >= 2:
        return 'Significant Structural Margin Issues'
    elif issues >= 1:
        return 'Moderate Margin Concerns'
    else:
        return 'Healthy'


division_df['Classification'] = division_df.apply(classify_division, axis=1)

summary_cols = [
    'Average Margin', 'Revenue', 'Profit Contribution',
    'Rev-to-Profit Ratio', 'Cost Efficiency',
    'Margin Efficiency Score', 'Classification'
]

print()
print(division_df[summary_cols].to_string())
print()
print("Legend:")
print("  Strong Financial Efficiency          — High margin, profit exceeds revenue share")
print("  Healthy                              — No significant issues detected")
print("  Moderate Margin Concerns             — Some inefficiency indicators present")
print("  Significant Structural Margin Issues — Multiple margin problems detected")

DIVISION FINANCIAL HEALTH SUMMARY

           Average Margin   Revenue  Profit Contribution  Rev-to-Profit Ratio  Cost Efficiency  Margin Efficiency Score                        Classification
Division                                                                                                                                                     
Chocolate        0.674483  0.928830             0.950577             1.023414         2.072036                 1.023414           Strong Financial Efficiency
Other            0.448446  0.068155             0.046375             0.680442         0.813061                 0.680442  Significant Structural Margin Issues
Sugar            0.666066  0.003015             0.003047             1.010643         1.994606                 1.010643           Strong Financial Efficiency

Legend:
  Strong Financial Efficiency          — High margin, profit exceeds revenue share
  Healthy                              — No significant issues detected
  Moderate 